# Brain stroke detection from CT — interactive lab

Train a **3D CNN** to detect stroke on brain CT, and use it to explore one idea hands-on:
**how data augmentation fights overfitting on a small dataset.**

You pick which augmentations to switch on, run a single training, and read the result:
training curves, ROC, the classification threshold, and the scans the model gets wrong.

Dataset: the cleaned brain-CT set published by Giovanni Pecoraro (Normal vs Stroke).
The data download and preprocessing are collapsed below — run them once and move on.

In [ ]:
#@title ⚙️ Setup, download and preprocessing (run once, then collapse)
import os, zipfile, math, warnings
warnings.filterwarnings("ignore")
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy import ndimage
from skimage import morphology
import imageio.v2 as imageio
from tqdm import tqdm
try:
    from google.colab import output as _co; _co.enable_custom_widget_manager()
except Exception:
    pass

# --- download the cleaned dataset -------------------------------------------
url = "https://github.com/Peco602/brain-stroke-detection-3d-cnn/releases/download/v0.0.1/brain_ct_data_fixed.zip"
path = tf.keras.utils.get_file("brain_ct_data_fixed.zip", url)
with zipfile.ZipFile(path, "r") as z: z.extractall(".")
NORMAL_PATH = "/content/brain_ct_data_fixed/Normal"
STROKE_PATH = "/content/brain_ct_data_fixed/Stroke"

# --- preprocessing helpers (denoise, resize, normalise, load) ---------------
def remove_noise(image):
    seg = morphology.dilation(image, np.ones((3,3)))
    seg[seg < 25] = 0; seg[seg > 25] = 1
    labels, _ = ndimage.label(seg)
    counts = np.bincount(labels.ravel().astype(int)); counts[0] = 0
    mask = labels == counts.argmax()
    mask = morphology.dilation(mask, np.ones((5,5)))
    mask = ndimage.binary_fill_holes(mask)
    mask = morphology.dilation(mask, np.ones((3,3)))
    return image * mask

def resize_scan(scan):
    dd, dw, dh = 64, 128, 128
    scan = ndimage.rotate(scan, 90, reshape=False)
    scan = ndimage.zoom(scan, (dw/scan.shape[0], dh/scan.shape[1], dd/scan.shape[-1]), order=1)
    return scan

def normalize_scan(scan):
    scan = np.clip(scan, 0, 255)
    return ((scan - 0) / 255).astype("float32")

def count_slices(path):
    d = {}
    for _, _, files in os.walk(path):
        for f in files:
            pid = int(f.split()[0]); d[pid] = d.get(pid, 0) + 1
    return d

def merge_slices(path, pid, n):
    return np.dstack(tuple(remove_noise(imageio.imread(f"{path}/{pid} ({s}).jpg"))
                           for s in range(1, n+1)))

def load_dataset(path):
    sd = count_slices(path)
    return np.array([resize_scan(normalize_scan(merge_slices(path, pid, n)))
                     for pid, n in tqdm(sd.items())])

print("Loading scans (denoise + resize)…")
normal_dataset = load_dataset(NORMAL_PATH)
stroke_dataset = load_dataset(STROKE_PATH)

# --- labels and 70/30 split -------------------------------------------------
normal_labels = np.zeros(len(normal_dataset), int)
stroke_labels = np.ones(len(stroke_dataset), int)
SPLIT = 0.7
n_tr = math.ceil(SPLIT*len(normal_labels)); s_tr = math.ceil(SPLIT*len(stroke_labels))
x_train = np.concatenate((normal_dataset[:n_tr], stroke_dataset[:s_tr]), 0)
y_train = np.concatenate((normal_labels[:n_tr], stroke_labels[:s_tr]), 0)
x_val   = np.concatenate((normal_dataset[n_tr:], stroke_dataset[s_tr:]), 0)
y_val   = np.concatenate((normal_labels[n_tr:], stroke_labels[s_tr:]), 0)

tf.random.set_seed(42)
batch_size = 2
training_dataset = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
                    .shuffle(len(x_train)).batch(batch_size).prefetch(2))
validation_dataset = (tf.data.Dataset.from_tensor_slices((x_val, y_val))
                      .shuffle(len(x_val)).batch(batch_size).prefetch(2))
print(f"Ready — {len(x_train)} training, {len(x_val)} validation scans "
      f"({int(y_train.sum())} stroke in train).")

In [ ]:
#@title 🧠 Model factory (run once) — 3D CNN with switchable augmentation
METRICS = [keras.metrics.BinaryAccuracy(name="accuracy"),
           keras.metrics.Precision(name="precision"),
           keras.metrics.Recall(name="recall"),
           keras.metrics.AUC(name="auc")]

def build_model(width=128, height=128, depth=64, lr=1e-4,
                rotation=False, flip=False, shift=False, zoom=False,
                brightness=False, contrast=False):
    """3D CNN. Each augmentation is a keras layer added only if its flag is on."""
    m = keras.Sequential()
    m.add(keras.Input((width, height, depth)))
    if rotation:   m.add(layers.RandomRotation(factor=(-0.125, 0.125), fill_mode="constant", fill_value=0))
    if flip:       m.add(layers.RandomFlip(mode="vertical"))
    if shift:      m.add(layers.RandomTranslation(0.2, 0.2, fill_mode="constant", fill_value=0))
    if zoom:       m.add(layers.RandomZoom(height_factor=0.15, fill_mode="constant", fill_value=0))
    if brightness: m.add(layers.RandomBrightness(factor=0.1, value_range=[0.0, 1.0]))
    if contrast:   m.add(layers.RandomContrast(factor=(0, 1.2)))
    m.add(layers.Reshape((width, height, depth, 1)))
    for f in (64, 64, 128, 256):
        m.add(layers.Conv3D(filters=f, kernel_size=3, activation="relu"))
        m.add(layers.MaxPool3D(pool_size=2))
        m.add(layers.BatchNormalization())
    m.add(layers.GlobalAveragePooling3D())
    m.add(layers.Dense(512, activation="relu"))
    m.add(layers.Dropout(0.3))
    m.add(layers.Dense(1, activation="sigmoid"))
    sched = keras.optimizers.schedules.ExponentialDecay(lr, 100000, 0.96, staircase=True)
    m.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=sched), metrics=METRICS)
    return m
print("build_model() ready.")

## Train with the augmentation you choose

Tick the augmentations you want, set the epochs, and press **Train**. Each run is added to
the comparison plot on the right, so you can see directly whether augmentation narrows the
gap between training and validation — the signature of less overfitting.

*3D training is slow: start with ~10 epochs. A GPU runtime is strongly recommended.*

In [ ]:
#@title 🎛 Augmentation & training
import time
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

STATE = {"model": None, "y_true": None, "y_score": None}
RUNS = {}   # label -> history, for the comparison plot

rot = W.Checkbox(value=False, description="rotation")
flp = W.Checkbox(value=False, description="flip")
shf = W.Checkbox(value=False, description="shift")
zm  = W.Checkbox(value=False, description="zoom")
brt = W.Checkbox(value=False, description="brightness")
ctr = W.Checkbox(value=False, description="contrast")
epochs_w = W.IntSlider(value=10, min=3, max=60, step=1, description="epochs", continuous_update=False)
lr_w     = W.Dropdown(options=[1e-4, 3e-4, 1e-3], value=1e-4, description="lr")
label_w  = W.Text(value="run 1", description="label")
btn = W.Button(description="Train", button_style="success", icon="play")
status = W.HTML()
out = W.Output()

def on_train(_):
    btn.disabled = True
    with out:
        clear_output(wait=True)
        try:
            keras.backend.clear_session()
            any_aug = any([rot.value, flp.value, shf.value, zm.value, brt.value, ctr.value])
            model = build_model(lr=lr_w.value, rotation=rot.value, flip=flp.value,
                                shift=shf.value, zoom=zm.value, brightness=brt.value,
                                contrast=ctr.value)
            on = [n for n, v in [("rot",rot.value),("flip",flp.value),("shift",shf.value),
                                 ("zoom",zm.value),("bright",brt.value),("contrast",ctr.value)] if v]
            print("Augmentation:", ", ".join(on) if on else "none")
            t1 = time.perf_counter()
            hist = model.fit(training_dataset, validation_data=validation_dataset,
                             epochs=epochs_w.value, verbose=2)
            secs = time.perf_counter() - t1
        except Exception as e:
            print("Training failed:", e); btn.disabled = False; return

        y_score = model.predict(validation_dataset, verbose=0).ravel()
        y_true  = np.concatenate([y.numpy() for _, y in validation_dataset], 0)
        fpr, tpr, _ = roc_curve(y_true, y_score); roc_auc = auc(fpr, tpr)
        STATE.update(model=model, y_true=y_true, y_score=y_score)
        RUNS[label_w.value or ("run %d" % (len(RUNS)+1))] = hist.history

        fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
        ax[0].plot(hist.history["auc"], label="train"); ax[0].plot(hist.history["val_auc"], label="valid")
        ax[0].set_title("AUC — this run"); ax[0].set_xlabel("epoch"); ax[0].set_ylim(0.2,1); ax[0].legend()
        for lbl, h in RUNS.items():
            ax[1].plot(h["val_auc"], label=lbl)
        ax[1].set_title("validation AUC — all runs"); ax[1].set_xlabel("epoch"); ax[1].set_ylim(0.2,1); ax[1].legend()
        plt.tight_layout(); plt.show()

        gap = hist.history["auc"][-1] - hist.history["val_auc"][-1]
        print("val AUC = {:.3f} | train-val AUC gap = {:.3f} | {:.0f}s".format(roc_auc, gap, secs))
        print("A large positive gap means the model overfits — try turning on more augmentation.")
        status.value = "<span style='color:green'>Done ✓ — threshold cell below is ready.</span>"
    btn.disabled = False

btn.on_click(on_train)
aug_box = W.VBox([W.HTML("<b>augmentation</b>"), rot, flp, shf, zm, brt, ctr])
cfg_box = W.VBox([W.HTML("<b>training</b>"), epochs_w, lr_w, label_w, btn, status])
display(W.HBox([W.VBox([aug_box, cfg_box]), out]))

## Tune the classification threshold

The network outputs a stroke probability; you choose the cut-off. Watch sensitivity,
specificity and the number of misclassified scans move as you slide it.

In [ ]:
#@title 🎚 Threshold vs sensitivity / specificity / misclassified
import ipywidgets as W
import matplotlib.pyplot as plt

def show_threshold(thr):
    if STATE["model"] is None:
        print("Train a model first."); return
    yt, ys = STATE["y_true"], STATE["y_score"]
    yp = (ys >= thr).astype(int)
    tp = int(((yp==1)&(yt==1)).sum()); tn = int(((yp==0)&(yt==0)).sum())
    fp = int(((yp==1)&(yt==0)).sum()); fn = int(((yp==0)&(yt==1)).sum())
    sens = tp/(tp+fn) if (tp+fn) else 0; spec = tn/(tn+fp) if (tn+fp) else 0
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    cm = np.array([[tn, fp],[fn, tp]])
    ax[0].imshow(cm, cmap="Purples")
    for (i,j), v in np.ndenumerate(cm):
        ax[0].text(j, i, str(v), ha="center", va="center", fontsize=16,
                   color="white" if v>cm.max()/2 else "black")
    ax[0].set_xticks([0,1]); ax[0].set_xticklabels(["pred normal","pred stroke"])
    ax[0].set_yticks([0,1]); ax[0].set_yticklabels(["true normal","true stroke"])
    ax[0].set_title("Confusion matrix @ %.2f" % thr)
    ax[1].hist(ys[yt==0], bins=12, alpha=.6, label="normal", color="steelblue")
    ax[1].hist(ys[yt==1], bins=12, alpha=.6, label="stroke", color="crimson")
    ax[1].axvline(thr, color="black", ls="--", lw=2, label="threshold")
    ax[1].set_xlabel("predicted probability of stroke"); ax[1].set_ylabel("scans")
    ax[1].set_title("Score distribution"); ax[1].legend()
    plt.tight_layout(); plt.show()
    print("Sensitivity = {:.2f}   Specificity = {:.2f}   Misclassified = {} of {}".format(
          sens, spec, fp+fn, len(yt)))
    print("TP={}  TN={}  FP={}  FN={}".format(tp, tn, fp, fn))

W.interact(show_threshold,
           thr=W.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="threshold",
                             continuous_update=False, layout=W.Layout(width="500px")))

## Inspect the misclassified scans

Step through the scans the model gets wrong and scroll the slices. Where does it fail —
small infarcts, artefacts, unusual anatomy?

In [ ]:
#@title 🔎 Browse the errors slice by slice
import ipywidgets as W
import matplotlib.pyplot as plt

def browse(thr, which, case, z):
    if STATE["model"] is None:
        print("Train a model first."); return
    yt, ys = STATE["y_true"], STATE["y_score"]
    yp = (ys >= thr).astype(int)
    if which == "false negatives": idxs = np.where((yp==0)&(yt==1))[0]
    elif which == "false positives": idxs = np.where((yp==1)&(yt==0))[0]
    else: idxs = np.where(yp==yt)[0]
    if len(idxs)==0: print("None of these at threshold %.2f." % thr); return
    k = idxs[case % len(idxs)]; vol = x_val[k]; z = min(z, vol.shape[2]-1)
    plt.figure(figsize=(5,5)); plt.imshow(vol[:,:,z], cmap="gray")
    plt.title("scan #{} — true={}  P(stroke)={:.2f}\nslice {}/{}".format(
              k, "stroke" if yt[k] else "normal", ys[k], z, vol.shape[2]-1))
    plt.axis("off"); plt.show()

W.interact(browse,
    thr=W.FloatSlider(value=0.5, min=0, max=1, step=0.02, description="threshold", continuous_update=False),
    which=W.Dropdown(options=["false negatives","false positives","correct"],
                     value="false negatives", description="show"),
    case=W.IntSlider(value=0, min=0, max=20, step=1, description="case #", continuous_update=False),
    z=W.IntSlider(value=32, min=0, max=63, step=1, description="slice", continuous_update=False))